## **FastF1 Data Aquisition**

#### **FastF1 References**

Installation: [https://docs.fastf1.dev/getting_started/installation.html](https://docs.fastf1.dev/getting_started/installation.html)</br>
API Documentation: [https://docs.fastf1.dev/api_reference/index.html](https://docs.fastf1.dev/api_reference/index.html)

##### **Imports**

In [ ]:
# Imports
import os

import fastf1 as ff1
import pandas as pd

from fastf1.req import RateLimitExceededError

#### **Event Data (FastF1)**

- Retrieve location data and start times for events from the years 2018-2025.
- Only 'Race' data selected, exlcudes Qualifiers.
- Merged with location data to create a single dataset with event information.


In [ ]:
event_times = []

# Get race starting times and locations for each year
for year in range(2018, 2026):
    event_schedule = ff1.get_event_schedule(year, include_testing=False)
    
    # Loop through season schedule, storing race info
    for _, event in event_schedule.iterrows():
        for i in range(1, 6):
            if event[f'Session{i}'] == 'Race':
                race_start = event[f'Session{i}DateUtc']
                break
        
        event_times.append({
            'EventName': event['EventName'],
            'Country': event['Country'],
            'Location': event['Location'],
            'StartTime': race_start,
            'Year': year
        })
        
event_data = pd.DataFrame(event_times)
    

#### **Location Data**

- Data retrieved from a GitHub repository **`f1-curcuits`** created by Tomo Bacinger
- Includes Longitude and Latitude of tracks from 1950 to present
- Location data will be used to aid in weather data collection for each track

Source: [https://github.com/bacinger/f1-circuits](https://github.com/bacinger/f1-circuits)<br>
Locations JSON: [https://github.com/bacinger/f1-circuits/blob/master/f1-locations.json](https://github.com/bacinger/f1-circuits/blob/master/f1-locations.json)

In [ ]:
# Import location data for f1 races
url = "https://raw.githubusercontent.com/bacinger/f1-circuits/master/f1-locations.json"

location_data = pd.read_json(url)[['location', 'lon', 'lat']]

# location_data.head()

In [ ]:
# Normalize location names across both data sources
location_data['location'] = location_data['location'].replace({
    'Montreal': 'Montréal',
    'Spa Francorchamps': 'Spa-Francorchamps',
    'Sao Paulo': 'São Paulo',
    'Nürburg': 'Nürburgring',
    'Scarperia e San Piero': 'Mugello'
})

event_data['Location'] = event_data['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

# Attach long, lat to each f1 race location
event_location_df = event_data.merge(location_data, left_on='Location', right_on='location', how='left')
event_location_df = event_location_df.drop(columns=['location'])

# Store location data
event_location_df.to_csv('../data/f1_event_location_data.csv', index=False)

#### **Race Data (FastF1)**

Race data for the 2018–2025 Formula 1 seasons was collected using the FastF1 Python package, which provides detailed lap-level, telemetry, and weather data for each race session. To simplify data management and minimize repeated API requests, the retrieved data was exported to yearly CSV files. FastF1 also supports local caching, allowing previously downloaded data to be stored and reused in future executions, though we have already stored the data in CSV files for future use.

**Notes**
- FastF1 request limit: **`500 requests per hour`**
- The collection process may require multiple runs to retrieve all race data.
- The function can be rerun and starts where it left off.

In [ ]:
# Arguements for different types of session data
data_config = {
    'laps': dict(laps=True, weather=False, telemetry=False, messages=False),
    'weather': dict(laps=False, weather=True, telemetry=False, messages=False)
}

In [ ]:
# Fetch race data from 2018-2025 for specified data types
def fetch_race_data(data_type):
    for year in range(2018, 2026):
        os.makedirs(f'../data/ff1_{data_type}_data', exist_ok=True) # create directory
        filepath = f'../data/ff1_{data_type}_data/{data_type}_data_{year}.csv'
        saved_events = []
            
        # Skip if already saved
        if os.path.exists(filepath):
            existing_race_data = pd.read_csv(filepath)
            saved_events = existing_race_data['EventName'].unique()
        
        try:
            event_schedule = ff1.get_event_schedule(year, include_testing=False)
        except RateLimitExceededError:
            print('ERROR: Rate Limit Exceeded (500 calls/hr). Please wait before rerunning.')
            return False
                
        #  Retrieve race data per event
        for event, location in zip(event_schedule['EventName'], event_schedule['Location']):
            if event in saved_events:
                print(f"Skipping {event} {year} — already saved")
                continue
            
            try:
                # Load session data with given config
                session = ff1.get_session(year, event, 'Race')
                session.load(**data_config[data_type])
                
                if data_type == 'laps':
                    rd = session.laps
                elif data_type == 'weather':
                    rd = session.weather_data
                
                # Save session info
                rd['Year'] = year           
                rd['Location'] = location 
                rd['EventName'] = event      
                rd.to_csv(filepath, mode='a', header=not os.path.exists(filepath), index=False)
            
            except RateLimitExceededError:
                print('ERROR: Rate Limit Exceeded (500 calls/hr). Please wait before rerunning.')
                return False
            
            except Exception as e: 
                print(f"ERROR: {e}")

In [ ]:
# Retrieve and store race data ('laps', 'weather')
fetch_race_data('weather')